# 09 — ALTitude × DepMap ALT Inventory (read-only diagnostic)

**Purpose.** Quantify how many RNA-backed DepMap osteosarcoma cell lines carry
a usable ALT call from ALTitude, and what class balance that yields for Aim 2.

The ~37 RNA-backed DepMap ModelIDs come from the Route B join established in
notebook 08. ALTitude `Curated.Marker` (Sheet1 of ALTitude-table1.xlsx) is the
gold-standard ALT call source. Gold calls from Taglialatela/Lawlor (already in
the manifest) take priority over ALTitude silver calls.

**Three reporting blocks:**
| Block | Scope | Positive arm definition |
|---|---|---|
| 1 | OS only | `alt_gold_positive` (gold-over-silver) |
| 2 | OS only — sensitivity | `alt_gold_positive` ∪ `alt_silver` |
| 3 | OS + non-OS | `alt_gold_positive` (flag SKLU1, TM31) |

**Read-only.** Does not modify the manifest, does not train models, does not load
the full expression matrix (ModelID index only). Writes
`data/processed/notebook_09_report.md`.

**Output:** `data/processed/notebook_09_report.md`

In [1]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import log_versions
import pandas as pd

log_versions()
print(f'Run: {RUN_TIMESTAMP}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  pyranges: 0.1.4
  openpyxl: 3.1.5
  requests: 2.33.1
  GEOparse: 2.0.4
Run: 2026-05-19T23:13:48Z


## 1 — Load manifest and shared DepMap lookups

In [2]:
from src.manifest_builder import load_altitude_cellines
from src.aim1_inventory import DEPMAP_RNA_PATH, depmap_rna_model_ids, WITHIN_COHORT_PER_ARM_MIN
from src.recovery_inventory import (
    MODEL_CSV_PATH,
    build_depmap_name_lookup,
    select_route_b_candidates,
    build_route_b_table,
)

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
print(f'Manifest rows: {len(manifest)}')

model_df = pd.read_csv(MODEL_CSV_PATH, dtype=str)
print(f'Model.csv rows: {len(model_df)}')

depmap_rna_models = depmap_rna_model_ids(DEPMAP_RNA_PATH)
print(f'DepMap RNA ModelIDs: {len(depmap_rna_models)}')

altitude = load_altitude_cellines()
altitude_ids = set(altitude['sample_id'].astype(str))
print(f'ALTitude panel rows: {len(altitude)}  unique sample_id: {len(altitude_ids)}')

Manifest rows: 115
Model.csv rows: 2154
DepMap RNA ModelIDs: 1719
ALTitude panel rows: 959  unique sample_id: 958


## 2 — Build Route B table → RNA-backed DepMap ModelIDs

Reuse the Route B join from notebook 08 to identify which DepMap ModelIDs
have expression data. This is the source of the ~37 RNA-backed model IDs.
ModelID index only — no full matrix load.

In [3]:
lookup = build_depmap_name_lookup(model_df)
route_b_candidates = select_route_b_candidates(manifest, altitude_ids)
route_b = build_route_b_table(route_b_candidates, lookup, model_df, depmap_rna_models)

# Unique RNA-backed ModelIDs from Route B
rna_backed = route_b[route_b['in_depmap_rna']]
rna_model_ids: set[str] = set(rna_backed['mapped_model_id'].dropna().astype(str))
print(f'Route B candidates:      {len(route_b_candidates)}')
print(f'Route B RNA-backed rows: {len(rna_backed)}')
print(f'Unique RNA-backed ModelIDs: {len(rna_model_ids)}')
print()

# Which model IDs are NOT in RNA? Log for completeness.
not_in_rna = route_b[~route_b['in_depmap_rna']]
if len(not_in_rna):
    print(f'!! {len(not_in_rna)} route_b row(s) excluded (not in DepMap RNA):')
    for _, r in not_in_rna.iterrows():
        print(f'   - {r["cell_line_name"]} → {r["mapped_model_id"]}  ({r["cohort"]})')

Route B candidates:      38
Route B RNA-backed rows: 37
Unique RNA-backed ModelIDs: 32

!! 1 route_b row(s) excluded (not in DepMap RNA):
   - SKLU1 → ACH-000309  (Taglialatela_2026)


## 3 — Join ALTitude calls and classify OS status

In [4]:
from src.altitude_depmap_inventory import (
    build_altitude_depmap_inventory,
    count_aim2_blocks,
    apply_altitude_depmap_verdict,
    NON_OS_KNOWN,
)

inv = build_altitude_depmap_inventory(manifest, altitude, model_df, rna_model_ids)

print(f'Inventory rows (one per unique RNA-backed ModelID): {len(inv)}')
print()
print('OS status distribution:')
print(inv['os_status'].value_counts().to_string())
print()
print('ALTitude coverage (altitude_alt_status):')
print(inv['altitude_alt_status'].value_counts().to_string())
print()
print('Final ALT status (gold-over-silver):')
print(inv['final_alt_status'].value_counts().to_string())
print()
print('Final provenance:')
print(inv['final_provenance'].value_counts().to_string())

Inventory rows (one per unique RNA-backed ModelID): 32

OS status distribution:
os_status
osteosarcoma    31
non_os           1

ALTitude coverage (altitude_alt_status):
altitude_alt_status
alt_gold_positive         12
alt_negative_high_conf     7
ambiguous_or_missing       7
alt_silver                 6

Final ALT status (gold-over-silver):
final_alt_status
alt_gold_positive         12
alt_negative_high_conf     7
ambiguous_or_missing       7
alt_silver                 6

Final provenance:
final_provenance
altitude_experimental       19
no_call                      7
altitude_prediction_only     6


## 4 — Log ambiguous or missing lines

Every line without a usable ALT call is logged explicitly.
These are excluded from all three reporting blocks.

In [5]:
ambiguous = inv[inv['final_alt_status'] == 'ambiguous_or_missing']
print(f'Ambiguous or missing ALT call: {len(ambiguous)} line(s)')
if len(ambiguous):
    display_cols = ['model_id', 'cell_line_name', 'os_status', 'manifest_alt_status',
                    'altitude_alt_status', 'altitude_evidence_tier', 'altitude_pred_class']
    with pd.option_context('display.max_rows', None, 'display.width', 200):
        print(ambiguous[display_cols].to_string(index=False))
else:
    print('None — all RNA-backed lines have a curated ALT call.')

print()
print('Disagreements (gold vs ALTitude):')
disagree = inv[inv['disagreement_note'] != '']
if len(disagree):
    for _, r in disagree.iterrows():
        print(f'  {r["model_id"]} ({r["cell_line_name"]}): {r["disagreement_note"]}')
else:
    print('None.')

Ambiguous or missing ALT call: 7 line(s)
  model_id cell_line_name    os_status manifest_alt_status  altitude_alt_status altitude_evidence_tier altitude_pred_class
ACH-000592           TM31       non_os         alt_unknown ambiguous_or_missing                unknown                    
ACH-000613            HOS osteosarcoma         alt_unknown ambiguous_or_missing           experimental                 nan
ACH-001275           SYO1 osteosarcoma         alt_unknown ambiguous_or_missing        prediction_only                ALT-
ACH-001813           C242 osteosarcoma         alt_unknown ambiguous_or_missing        prediction_only                ALT-
ACH-001817           C393 osteosarcoma         alt_unknown ambiguous_or_missing        prediction_only                ALT-
ACH-002779     NCCESOS1C1 osteosarcoma         alt_unknown ambiguous_or_missing        prediction_only                ALT-
ACH-003176        NRHGCT1 osteosarcoma         alt_unknown ambiguous_or_missing        prediction_

## 5 — Block 1: OS-only, gold-over-silver

Primary contrast for Aim 2.
Positive arm = `alt_gold_positive` only (manifest gold or ALTitude high-confidence).  
Negative arm = `alt_negative_high_conf`.

In [6]:
os_inv = inv[inv['os_status'] == 'osteosarcoma'].copy()

print(f'=== Block 1: OS-only, gold-over-silver ({len(os_inv)} OS lines total) ===')
print(f'  ALT-positive (gold):   {(os_inv["final_alt_status"] == "alt_gold_positive").sum()}')
print(f'  ALT-negative (gold):   {(os_inv["final_alt_status"] == "alt_negative_high_conf").sum()}')
print(f'  ALT-silver:            {(os_inv["final_alt_status"] == "alt_silver").sum()}  (not in this block)')
print(f'  Ambiguous/missing:     {(os_inv["final_alt_status"] == "ambiguous_or_missing").sum()}')
print()

b1_pos = os_inv[os_inv['final_alt_status'] == 'alt_gold_positive']
b1_neg = os_inv[os_inv['final_alt_status'] == 'alt_negative_high_conf']

print(f'ALT-positive ({len(b1_pos)}):')
for _, r in b1_pos.iterrows():
    print(f'  {r["model_id"]:15s}  {r["cell_line_name"]:20s}  prov={r["final_provenance"]}')
print()
print(f'ALT-negative ({len(b1_neg)}):')
for _, r in b1_neg.iterrows():
    print(f'  {r["model_id"]:15s}  {r["cell_line_name"]:20s}  prov={r["final_provenance"]}')
print()
print(f'min(pos, neg) = {min(len(b1_pos), len(b1_neg))}  (within-cohort minimum = {WITHIN_COHORT_PER_ARM_MIN})')

=== Block 1: OS-only, gold-over-silver (31 OS lines total) ===
  ALT-positive (gold):   12
  ALT-negative (gold):   7
  ALT-silver:            6  (not in this block)
  Ambiguous/missing:     6

ALT-positive (12):
  ACH-000082       G292CLONEA141B1       prov=altitude_experimental
  ACH-000364       U2OS                  prov=altitude_experimental
  ACH-000410       SAOS2                 prov=altitude_experimental
  ACH-001526       HUO9                  prov=altitude_experimental
  ACH-001715       CAL72                 prov=altitude_experimental
  ACH-002067       NOS1                  prov=altitude_experimental
  ACH-002178       NY                    prov=altitude_experimental
  ACH-002471       OS052                 prov=altitude_experimental
  ACH-002688       OS457                 prov=altitude_experimental
  ACH-002689       OS525                 prov=altitude_experimental
  ACH-002690       OS526                 prov=altitude_experimental
  ACH-002834       OS384               

## 6 — Block 2: OS-only, ALTitude-silver sensitivity

Sensitivity analysis: include `alt_silver` in the positive arm.  
Silver labels carry higher label noise; this block answers whether including
them would change the verdict.

In [7]:
print(f'=== Block 2: OS-only, silver sensitivity ({len(os_inv)} OS lines) ===')
b2_pos_mask = os_inv['final_alt_status'].isin(['alt_gold_positive', 'alt_silver'])
b2_neg_mask = os_inv['final_alt_status'] == 'alt_negative_high_conf'
print(f'  ALT-positive (gold+silver): {b2_pos_mask.sum()}')
print(f'    of which gold:            {(os_inv["final_alt_status"] == "alt_gold_positive").sum()}')
print(f'    of which silver:          {(os_inv["final_alt_status"] == "alt_silver").sum()}')
print(f'  ALT-negative:               {b2_neg_mask.sum()}')
print(f'  Ambiguous/missing:          {(os_inv["final_alt_status"] == "ambiguous_or_missing").sum()}')
print()

b2_silver_only = os_inv[os_inv['final_alt_status'] == 'alt_silver']
if len(b2_silver_only):
    print(f'Silver-only additions to positive arm ({len(b2_silver_only)}):')
    for _, r in b2_silver_only.iterrows():
        print(f'  {r["model_id"]:15s}  {r["cell_line_name"]:20s}  '
              f'pred_class={r["altitude_pred_class"]}  tier={r["altitude_evidence_tier"]}  prob={r["altitude_alt_prob"]:.4f}')
print()
print(f'min(pos, neg) = {min(b2_pos_mask.sum(), b2_neg_mask.sum())}  '
      f'(within-cohort minimum = {WITHIN_COHORT_PER_ARM_MIN})')

=== Block 2: OS-only, silver sensitivity (31 OS lines) ===
  ALT-positive (gold+silver): 18
    of which gold:            12
    of which silver:          6
  ALT-negative:               7
  Ambiguous/missing:          6

Silver-only additions to positive arm (6):
  ACH-002433       CCLFPEDS0019T         pred_class=ALT+  tier=prediction_only  prob=1.0000
  ACH-003171       NRHOS1                pred_class=ALT+  tier=prediction_only  prob=1.0000
  ACH-003178       NRHOS2                pred_class=ALT+  tier=prediction_only  prob=1.0000
  ACH-003185       NRHOS4                pred_class=ALT+  tier=prediction_only  prob=1.0000
  ACH-003573       C337                  pred_class=ALT+  tier=prediction_only  prob=1.0000
  ACH-003587       531MII                pred_class=ALT+  tier=prediction_only  prob=1.0000

min(pos, neg) = 7  (within-cohort minimum = 3)


## 7 — Block 3: OS + non-OS

All RNA-backed DepMap lines (any OncotreePrimaryDisease), gold-over-silver.
SKLU1 (lung, ACH-000309) and TM31 (glioblastoma, ACH-000592) are explicitly
flagged — they must not be included in an OS-restricted analysis.

In [8]:
print(f'=== Block 3: OS + non-OS, gold-over-silver ({len(inv)} total lines) ===')
print(f'  ALT-positive (gold): {(inv["final_alt_status"] == "alt_gold_positive").sum()}')
print(f'  ALT-negative (gold): {(inv["final_alt_status"] == "alt_negative_high_conf").sum()}')
print(f'  ALT-silver:          {(inv["final_alt_status"] == "alt_silver").sum()}')
print(f'  Ambiguous/missing:   {(inv["final_alt_status"] == "ambiguous_or_missing").sum()}')
print()

non_os_lines = inv[inv['os_status'] != 'osteosarcoma']
if len(non_os_lines):
    print(f'!! Non-OS lines in RNA-backed set ({len(non_os_lines)}) — MUST NOT be included in OS analysis:')
    for _, r in non_os_lines.iterrows():
        known_flag = NON_OS_KNOWN.get(r['model_id'], r['os_status'])
        print(f'   - {r["model_id"]}  {r["cell_line_name"]:20s}  disease: {known_flag}  '
              f'final_alt={r["final_alt_status"]}')

# SKLU1 is NOT in DepMap RNA so it won't appear in inv — log that explicitly
sklu1_id = 'ACH-000309'
if sklu1_id not in rna_model_ids:
    print(f'\nNote: SKLU1 ({sklu1_id}) is NOT in DepMap RNA matrix — excluded from all blocks.')
    print('  ALTitude has a call for SKLU1 but it cannot be used without RNA backing.')

=== Block 3: OS + non-OS, gold-over-silver (32 total lines) ===
  ALT-positive (gold): 12
  ALT-negative (gold): 7
  ALT-silver:          6
  Ambiguous/missing:   7

!! Non-OS lines in RNA-backed set (1) — MUST NOT be included in OS analysis:
   - ACH-000592  TM31                  disease: glioblastoma (TM31)  final_alt=ambiguous_or_missing

Note: SKLU1 (ACH-000309) is NOT in DepMap RNA matrix — excluded from all blocks.
  ALTitude has a call for SKLU1 but it cannot be used without RNA backing.


## 8 — Crosstab and provenance summary

In [9]:
blocks = count_aim2_blocks(inv)

print('=== Block counts summary ===')
print(f"{'Block':<35} {'pos':>5} {'neg':>5} {'amb':>5} {'min':>5}")
print('-' * 55)
for bname, bc in blocks.items():
    print(f"{bname:<35} {bc['n_positive']:>5} {bc['n_negative']:>5} "
          f"{bc['n_ambiguous_or_missing']:>5} {bc['min_arm']:>5}")
print(f'  (within-cohort minimum per arm = {WITHIN_COHORT_PER_ARM_MIN})')
print()

# Per-provenance breakdown
print('=== Provenance breakdown (OS lines only) ===')
prov_tab = os_inv.groupby(['final_provenance', 'final_alt_status']).size().unstack(fill_value=0)
with pd.option_context('display.max_columns', None):
    print(prov_tab.to_string())

=== Block counts summary ===
Block                                 pos   neg   amb   min
-------------------------------------------------------
block1_os_gold                         12     7     6     7
block2_os_sensitivity                  18     7     6     7
block3_all_gold                        12     7     7     7
  (within-cohort minimum per arm = 3)

=== Provenance breakdown (OS lines only) ===
final_alt_status          alt_gold_positive  alt_negative_high_conf  alt_silver  ambiguous_or_missing
final_provenance                                                                                     
altitude_experimental                    12                       7           0                     0
altitude_prediction_only                  0                       0           6                     0
no_call                                   0                       0           0                     6


## 9 — AIM 2 VIA ALTITUDE-DEPMAP VERDICT

In [10]:
verdict = apply_altitude_depmap_verdict(blocks, per_arm_min=WITHIN_COHORT_PER_ARM_MIN)

VERDICT_LABEL = {
    'within_cohort_viable':    'WITHIN-COHORT VIABLE',
    'pooled_exploratory_only': 'POOLED-EXPLORATORY ONLY',
    'not_estimable':           'NOT ESTIMABLE AS DEFINED',
}

print('=' * 60)
print(f"AIM 2 VIA ALTITUDE-DEPMAP: {VERDICT_LABEL[verdict['category']]}")
print('=' * 60)
print(verdict['justification'])
print()
if verdict['category'] == 'within_cohort_viable':
    print(f"  Positive arm: {verdict['n_positive']}  "
          f"Negative arm: {verdict['n_negative']}")
elif verdict['category'] == 'pooled_exploratory_only':
    print(f"  Positive arm (incl. silver): {verdict['n_positive']}  "
          f"Negative arm: {verdict['n_negative']}  "
          f"Silver added: {verdict['n_silver_added']}")
else:
    print(f"  Gold — pos: {verdict['n_positive_gold']}  neg: {verdict['n_negative_gold']}")
    print(f"  Gold+silver — pos: {verdict['n_positive_silver']}  "
          f"neg: {verdict['n_negative_silver']}")

AIM 2 VIA ALTITUDE-DEPMAP: WITHIN-COHORT VIABLE
OS-only gold labels: 12 ALT-positive and 7 ALT-negative RNA-backed DepMap lines. Both arms ≥ 3; all from the same cohort/platform (DepMap 26Q1 log2TPM). Sufficient for within-cohort Aim 2 analysis.

  Positive arm: 12  Negative arm: 7


## 10 — Write `data/processed/notebook_09_report.md`

In [11]:
REPORT_PATH = ROOT / 'data' / 'processed' / 'notebook_09_report.md'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

VERDICT_LABEL = {
    'within_cohort_viable':    'WITHIN-COHORT VIABLE',
    'pooled_exploratory_only': 'POOLED-EXPLORATORY ONLY',
    'not_estimable':           'NOT ESTIMABLE AS DEFINED',
}

lines: list[str] = []
lines.append('# Notebook 09 — ALTitude × DepMap ALT Inventory\n')
lines.append(f'_Generated: {RUN_TIMESTAMP}_\n')
lines.append('')
lines.append(
    'Read-only diagnostic quantifying Aim 2 class balance from ALTitude curated '
    'ALT calls for RNA-backed DepMap OS cell lines.\n'
)
lines.append('')

lines.append('## Data loading\n')
lines.append(f'- Manifest rows: **{len(manifest)}**\n')
lines.append(f'- DepMap RNA ModelIDs (index only): **{len(depmap_rna_models)}**\n')
lines.append(f'- ALTitude panel rows: **{len(altitude)}**  unique ModelIDs: **{len(altitude_ids)}**\n')
lines.append(f'- Route B RNA-backed unique ModelIDs: **{len(rna_model_ids)}**\n')
lines.append('')

lines.append('## Non-OS lines in RNA-backed set\n')
non_os_lines = inv[inv['os_status'] != 'osteosarcoma']
if len(non_os_lines):
    for _, r in non_os_lines.iterrows():
        known = NON_OS_KNOWN.get(r['model_id'], r['os_status'])
        lines.append(
            f'- **{r["model_id"]}** ({r["cell_line_name"]}): {known} — '
            f'final_alt={r["final_alt_status"]}. **Excluded from OS blocks.**\n'
        )
else:
    lines.append('None — all RNA-backed lines are osteosarcoma.\n')

sklu1_id = 'ACH-000309'
if sklu1_id not in rna_model_ids:
    lines.append(
        '- **ACH-000309** (SKLU1): lung_carcinoma — maps via Route B but '
        '**absent from DepMap RNA matrix**; excluded from all blocks.\n'
    )
lines.append('')

lines.append('## Ambiguous or missing ALT calls\n')
ambiguous = inv[inv['final_alt_status'] == 'ambiguous_or_missing']
if len(ambiguous):
    for _, r in ambiguous.iterrows():
        lines.append(
            f'- **{r["model_id"]}** ({r["cell_line_name"]}): '
            f'manifest={r["manifest_alt_status"]}, altitude={r["altitude_alt_status"]}\n'
        )
else:
    lines.append('None — all RNA-backed lines have a curated ALT call.\n')
lines.append('')

lines.append('## Disagreements (gold vs ALTitude)\n')
disagree = inv[inv['disagreement_note'] != '']
if len(disagree):
    for _, r in disagree.iterrows():
        lines.append(f'- **{r["model_id"]}** ({r["cell_line_name"]}): {r["disagreement_note"]}\n')
else:
    lines.append('None.\n')
lines.append('')

lines.append('## Block counts summary\n')
lines.append('| Block | n_positive | n_negative | n_ambiguous | min_arm |\n')
lines.append('|---|---|---|---|---|\n')
for bname, bc in blocks.items():
    lines.append(
        f"| {bname} | {bc['n_positive']} | {bc['n_negative']} | "
        f"{bc['n_ambiguous_or_missing']} | {bc['min_arm']} |\n"
    )
lines.append(f'_Within-cohort minimum per arm = {WITHIN_COHORT_PER_ARM_MIN}_\n')
lines.append('')

lines.append('## Provenance breakdown (OS lines only)\n')
prov_tab = os_inv.groupby(['final_provenance', 'final_alt_status']).size().unstack(fill_value=0)
lines.append(prov_tab.to_markdown() + '\n')
lines.append('')

lines.append('## AIM 2 VIA ALTITUDE-DEPMAP VERDICT\n')
lines.append(f'**{VERDICT_LABEL[verdict["category"]]}**\n')
lines.append('')
lines.append(f'> {verdict["justification"]}\n')
lines.append('')
if verdict['category'] == 'within_cohort_viable':
    lines.append(
        f'Positive arm: **{verdict["n_positive"]}**  '
        f'Negative arm: **{verdict["n_negative"]}**  '
        f'Block: {verdict["block"]}\n'
    )
elif verdict['category'] == 'pooled_exploratory_only':
    lines.append(
        f'Positive arm (incl. silver): **{verdict["n_positive"]}**  '
        f'Negative arm: **{verdict["n_negative"]}**  '
        f'Silver added: {verdict["n_silver_added"]}  '
        f'Block: {verdict["block"]}\n'
    )
else:
    lines.append(
        f'Gold — pos: {verdict["n_positive_gold"]} neg: {verdict["n_negative_gold"]}  '
        f'Gold+silver — pos: {verdict["n_positive_silver"]} neg: {verdict["n_negative_silver"]}\n'
    )
lines.append('')
lines.append('## Next steps\n')
lines.append('1. Review and accept this class-balance report.\n')
lines.append('2. If verdict is WITHIN-COHORT VIABLE, proceed to Aim 2 ALT classifier design.\n')
lines.append('3. If verdict is POOLED-EXPLORATORY or NOT ESTIMABLE, request Route A/C data.\n')
lines.append('4. No DE or modeling code has been written.\n')
lines.append('')

REPORT_PATH.write_text(''.join(lines))
print(f'Wrote {REPORT_PATH.relative_to(ROOT)}  ({REPORT_PATH.stat().st_size} bytes)')

Wrote data/processed/notebook_09_report.md  (2769 bytes)


## 11 — Idempotence check

In [12]:
import hashlib

first_hash = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
REPORT_PATH.write_text(''.join(lines))
second_hash = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
assert first_hash == second_hash, 'Notebook 09 is not idempotent'
print('Idempotent: re-writing produces identical bytes.')

Idempotent: re-writing produces identical bytes.


In [13]:
import pandas as pd
from pathlib import Path

ROOT = Path('.').resolve()
ROOT = ROOT if (ROOT / 'src').is_dir() else ROOT.parent

# Find the ALTitude table file
alt_path = ROOT / 'data' / 'raw' / 'literature_tables'
for f in alt_path.glob('*ALTitude*'):
    print(f.name)
for f in alt_path.glob('*altitude*'):
    print(f.name)

ALTitude-table1.xlsx
ALTitude-table2,4,5.xlsx
ALTitude-table2,4,5-sh1.csv
ALTitude-table2,4,5-sh2.csv
ALTitude-table2,4,5-sh3.csv
ALTitude-table3.csv
ALTitude-table3.xlsx


In [15]:
# adjust filename to whatever the glob above found
xl = pd.ExcelFile(alt_path / 'ALTitude-table1.xlsx')
print("Sheets:", xl.sheet_names)
df = pd.read_excel(alt_path / 'ALTitude-table1.xlsx', sheet_name='Sheet1')
print("\nColumns:")
for c in df.columns:
    print(f"  {c!r}")
print("\nFirst 3 rows, all columns:")
with pd.option_context('display.max_columns', None, 'display.width', 250):
    print(df.head(3).to_string())

Sheets: ['Sheet1']

Columns:
  'Achilles'
  'StripCI'
  'TelSeq'
  'TelFusDetector_rate'
  'TelFuse_newtelomere'
  'AAAGGG'
  'AACGGG'
  'AATGGG'
  'ACAGGG'
  'ACCGGG'
  'ACTGGG'
  'AGAGGG'
  'AGCGGG'
  'AGTGGG'
  'ATAGGG'
  'ATCGGG'
  'ATTGGG'
  'CAAGGG'
  'CACGGG'
  'CATGGG'
  'CCAGGG'
  'CCCGGG'
  'CCTGGG'
  'CGAGGG'
  'CGCGGG'
  'CGTGGG'
  'CTAGGG'
  'CTCGGG'
  'CTTGGG'
  'GAAGGG'
  'GACGGG'
  'GATGGG'
  'GCAGGG'
  'GCCGGG'
  'GCTGGG'
  'GGAGGG'
  'GGCGGG'
  'GGGGGG'
  'GGTGGG'
  'GTAGGG'
  'GTCGGG'
  'GTTGGG'
  'TAAGGG'
  'TACGGG'
  'TATGGG'
  'TCAGGG'
  'TCCGGG'
  'TCTGGG'
  'TGAGGG'
  'TGCGGG'
  'TGTGGG'
  'TTAGGG'
  'TTCGGG'
  'TTTGGG'
  'TERT_dep'
  'ATRX_dep'
  'TERT_exp'
  'Sex'
  'OncotreePrimaryDisease'
  'OncotreeLineage'
  'PatientRace'
  'PrimaryOrMetastasis'
  'TVR'
  'ALT_pred_class'
  'ALT_pred_prob'
  'SMARCAL1_exp'
  'Age'
  'SMARCAL1_dep'
  'Original.Marker'
  'Curated.Marker'
  'ATRX_Marker.Type'
  'ATRX_Exon'
  'ATRX_RefSeq'
  'ATRX_Region'
  'ATRX_Class'
  'AAC

In [16]:
import pandas as pd
from pathlib import Path

ROOT = Path('.').resolve()
ROOT = ROOT if (ROOT / 'src').is_dir() else ROOT.parent
alt_path = ROOT / 'data' / 'raw' / 'literature_tables'

df = pd.read_excel(alt_path / 'ALTitude-table1.xlsx', sheet_name='Sheet1')

# What does the ALT call column actually contain, globally?
print("ALT_pred_class value counts (all 958 lines):")
print(df['ALT_pred_class'].value_counts(dropna=False).to_string())
print()

# The 31 OS DepMap ModelIDs your notebook already identified as RNA-backed.
# ALTitude's ModelID is the first column 'Achilles'.
rna_backed_os = [
    'ACH-000082','ACH-000359','ACH-000364','ACH-000410','ACH-000613',
    'ACH-000748','ACH-001001','ACH-001275','ACH-001526','ACH-001715',
    'ACH-001813','ACH-001814','ACH-001817','ACH-002067','ACH-002069',
    'ACH-002142','ACH-002178','ACH-002433','ACH-002470','ACH-002471',
    'ACH-002688','ACH-002689','ACH-002690','ACH-002779','ACH-002834',
    'ACH-003171','ACH-003176','ACH-003178','ACH-003185','ACH-003573',
    'ACH-003587',
]

sub = df[df['Achilles'].isin(rna_backed_os)].copy()
print(f"Matched {len(sub)} of 31 RNA-backed OS lines in ALTitude")
print()
cols = ['Achilles','StripCI','ALT_pred_class','ALT_pred_prob',
        'ALT source','ALT_assay type','ALT assay strength',
        'OncotreePrimaryDisease']
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 40, 'display.width', 250):
    print(sub[cols].to_string(index=False))
print()
print("ALT_pred_class among the 31 RNA-backed OS lines:")
print(sub['ALT_pred_class'].value_counts(dropna=False).to_string())
print()
print("How many have experimental ALT evidence (ALT source non-null):")
print(sub['ALT source'].notna().sum())

ALT_pred_class value counts (all 958 lines):
ALT_pred_class
ALT-    893
ALT+     39
NaN      27

Matched 31 of 31 RNA-backed OS lines in ALTitude

  Achilles       StripCI ALT_pred_class  ALT_pred_prob                                                                                                                                          ALT source                                    ALT_assay type         ALT assay strength OncotreePrimaryDisease
ACH-003573          C337           ALT+   1.000000e+00                                                                                                                                                 NaN                                               NaN                        NaN           Osteosarcoma
ACH-001715         CAL72           ALT+   1.000000e+00                                                                                                                                    Deeg et al. 2017                           c-circles and APB 